In [1]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/AtualizaIndicador


#### Busca Dados INPC via código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.inpc import Inpc

with conectar():
    dados_inpc = Inpc.buscar()
    for aa in dados_inpc:
        print(aa)


#### Busca Dados INPC via código Atualiza-Indicador

In [ ]:

from database.conexao import conectar
from indicadores.inpc import Inpc

with conectar():
    dados_inpc = Inpc.select()
    for aa in dados_inpc:
        print(aa.indice)
        print(aa.dt_referencia)


#### Busca Dados INPC Diretamente da API

In [ ]:

ultimos_meses = 6

# API Banco Central -------------------------------------------
#url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados/ultimos/{ultimos_meses}?formato=json"
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.188/dados/ultimos/{ultimos_meses}?formato=json"

#parametros = { "formato": "json", "dataInicial": data_inicial, "dataFinal": data_final }

resposta = requests.get(url, timeout=10)
resposta.raise_for_status()
dados = resposta.json() # lista de dicionários

# Percorrer a lista de dicionário
for inpc in dados:
    print(inpc)


#### Gera datas

In [ ]:

# Datas -------------------------------------------------------
hoje = datetime.now()

data_inicial = (hoje - timedelta(days=230)).strftime("%d/%m/%Y")
data_final = hoje.strftime("%d/%m/%Y")

print(f"Período: {data_inicial} até {data_final}")


#### Busca Dados INPC da API com parêmetro de datas

In [ ]:

# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.188/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status() # Valida a resposta HTTP; se houver erro, não deixa o código continuar.
dados = resposta.json()     # lista de dicionários

# Percorrer a lista de dicionário
for inpc in dados:
    print(inpc)


#### Insere Dados no Banco (transformando dados)

In [ ]:
# API Banco Central -------------------------------------------
url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.188/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"

resposta = requests.get(url, timeout=10)
resposta.raise_for_status() # Valida a resposta HTTP; se houver erro, não deixa o código continuar.
dados = resposta.json()     # lista de dicionários

# Transforma dados para o formato da tabela Igpm
dados_inpc = []
for registro in dados:
    dados_inpc.append({"indice": registro["valor"],
                       "status": True,
                       "dt_referencia": datetime.strptime(registro["data"], "%d/%m/%Y").date()
                      })

with conectar():
    Inpc.insert_many(dados_inpc).on_conflict_ignore().execute()

#### Visualiza DADOS INPC (lista de dicionário)

In [ ]:
for inpc in dados:
    print(inpc["data"], inpc["valor"])


#### Carga direta no Banco (registro por registro)

In [ ]:

from database.conexao import conectar
from indicadores.inpc import Inpc
from decimal import Decimal

with conectar():
    for registro in dados:
        Inpc.insert(indice=registro["valor"], status=True, dt_referencia=registro["data"]).on_conflict_ignore().execute()


#### Carga direta no Banco (Em Massa)

In [ ]:

from database.conexao import conectar
from indicadores.inpc import Inpc
from decimal import Decimal

with conectar():
    Inpc.insert_many(dados_inpc).on_conflict_ignore().execute()

#### Atualiza tabela INPC usando método do código Atualiza-Indicador

In [3]:

from database.conexao import conectar
from indicadores.inpc import Inpc

aa = Inpc.atualizar_inpc()

print(aa)

None


#### Carga Histórica

In [4]:

import requests
from datetime import date, datetime, timedelta
from decimal import Decimal
from time import sleep
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from database.conexao import conectar
from indicadores.indicador import Indicador, IndicadorInfo


def carga_historica(nome, serie, ano_inicial=1980):
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
    inicio = date(ano_inicial, 1, 1)
    hoje = date.today()

    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])

    with requests.Session() as sessao:
        sessao.mount("https://", HTTPAdapter(max_retries=retry))

        with conectar():
            cd_indicador = IndicadorInfo.get(IndicadorInfo.nm_indicador == nome).cd_indicador

        total = 0

        while inicio <= hoje:
            fim = min(date(inicio.year, 12, 31), hoje)
            parametros = {
                "formato": "json",
                "dataInicial": inicio.strftime("%d/%m/%Y"),
                "dataFinal": fim.strftime("%d/%m/%Y")
            }

            resposta = sessao.get(url, params=parametros, timeout=30)
            resposta.raise_for_status()

            registros = [{
                "cd_indicador": cd_indicador,
                "valor": Decimal(item["valor"]),
                "status": True,
                "dt_referencia": datetime.strptime(item["data"], "%d/%m/%Y").date()
            } for item in resposta.json()]

            if registros:
                with conectar():
                    Indicador.insert_many(registros).on_conflict_ignore().execute()

            total += len(registros)
            print(f"{nome} {inicio.year}: {len(registros)} registros")
            inicio = fim + timedelta(days=1)
            sleep(1)

    print(f"\n{nome}: {total} registros consultados")
    

In [5]:
carga_historica("INPC", 188)

INPC 1980: 12 registros
INPC 1981: 12 registros
INPC 1982: 12 registros
INPC 1983: 12 registros
INPC 1984: 12 registros
INPC 1985: 12 registros
INPC 1986: 12 registros
INPC 1987: 12 registros
INPC 1988: 12 registros
INPC 1989: 12 registros
INPC 1990: 12 registros
INPC 1991: 12 registros
INPC 1992: 12 registros
INPC 1993: 12 registros
INPC 1994: 12 registros
INPC 1995: 12 registros
INPC 1996: 12 registros
INPC 1997: 12 registros
INPC 1998: 12 registros
INPC 1999: 12 registros
INPC 2000: 12 registros
INPC 2001: 12 registros
INPC 2002: 12 registros
INPC 2003: 12 registros
INPC 2004: 12 registros
INPC 2005: 12 registros
INPC 2006: 12 registros
INPC 2007: 12 registros
INPC 2008: 12 registros
INPC 2009: 12 registros
INPC 2010: 12 registros
INPC 2011: 12 registros
INPC 2012: 12 registros
INPC 2013: 12 registros
INPC 2014: 12 registros
INPC 2015: 12 registros
INPC 2016: 12 registros
INPC 2017: 12 registros
INPC 2018: 12 registros
INPC 2019: 12 registros
INPC 2020: 12 registros
INPC 2021: 12 re

In [7]:
carga_historica("IGPM", 189, 1989)

IGPM 1989: 7 registros
IGPM 1990: 12 registros
IGPM 1991: 12 registros
IGPM 1992: 12 registros
IGPM 1993: 12 registros
IGPM 1994: 12 registros
IGPM 1995: 12 registros
IGPM 1996: 12 registros
IGPM 1997: 12 registros
IGPM 1998: 12 registros
IGPM 1999: 12 registros
IGPM 2000: 12 registros
IGPM 2001: 12 registros
IGPM 2002: 12 registros
IGPM 2003: 12 registros
IGPM 2004: 12 registros
IGPM 2005: 12 registros
IGPM 2006: 12 registros
IGPM 2007: 12 registros
IGPM 2008: 12 registros
IGPM 2009: 12 registros
IGPM 2010: 12 registros
IGPM 2011: 12 registros
IGPM 2012: 12 registros
IGPM 2013: 12 registros
IGPM 2014: 12 registros
IGPM 2015: 12 registros
IGPM 2016: 12 registros
IGPM 2017: 12 registros
IGPM 2018: 12 registros
IGPM 2019: 12 registros
IGPM 2020: 12 registros
IGPM 2021: 12 registros
IGPM 2022: 12 registros
IGPM 2023: 12 registros
IGPM 2024: 12 registros
IGPM 2025: 12 registros
IGPM 2026: 9 registros

IGPM: 448 registros consultados


In [9]:
carga_historica("CDI", 4391, 1986)

CDI 1986: 5 registros
CDI 1987: 12 registros
CDI 1988: 12 registros
CDI 1989: 12 registros
CDI 1990: 12 registros
CDI 1991: 12 registros
CDI 1992: 12 registros
CDI 1993: 12 registros
CDI 1994: 12 registros
CDI 1995: 12 registros
CDI 1996: 12 registros
CDI 1997: 12 registros
CDI 1998: 12 registros
CDI 1999: 12 registros
CDI 2000: 12 registros
CDI 2001: 12 registros
CDI 2002: 12 registros
CDI 2003: 12 registros
CDI 2004: 12 registros
CDI 2005: 12 registros
CDI 2006: 12 registros
CDI 2007: 12 registros
CDI 2008: 12 registros
CDI 2009: 12 registros
CDI 2010: 12 registros
CDI 2011: 12 registros
CDI 2012: 12 registros
CDI 2013: 12 registros
CDI 2014: 12 registros
CDI 2015: 12 registros
CDI 2016: 12 registros
CDI 2017: 12 registros
CDI 2018: 12 registros
CDI 2019: 12 registros
CDI 2020: 12 registros
CDI 2021: 12 registros
CDI 2022: 12 registros
CDI 2023: 12 registros
CDI 2024: 12 registros
CDI 2025: 12 registros
CDI 2026: 10 registros

CDI: 483 registros consultados


In [10]:
carga_historica("TR", 7811, 1991)

TR 1991: 10 registros
TR 1992: 12 registros
TR 1993: 12 registros
TR 1994: 12 registros
TR 1995: 12 registros
TR 1996: 12 registros
TR 1997: 12 registros
TR 1998: 12 registros
TR 1999: 12 registros
TR 2000: 12 registros
TR 2001: 12 registros
TR 2002: 12 registros
TR 2003: 12 registros
TR 2004: 12 registros
TR 2005: 12 registros
TR 2006: 12 registros
TR 2007: 12 registros
TR 2008: 12 registros
TR 2009: 12 registros
TR 2010: 12 registros
TR 2011: 12 registros
TR 2012: 12 registros
TR 2013: 12 registros
TR 2014: 12 registros
TR 2015: 12 registros
TR 2016: 12 registros
TR 2017: 12 registros
TR 2018: 12 registros
TR 2019: 12 registros
TR 2020: 12 registros
TR 2021: 12 registros
TR 2022: 12 registros
TR 2023: 12 registros
TR 2024: 12 registros
TR 2025: 12 registros
TR 2026: 10 registros

TR: 428 registros consultados


In [11]:
import requests
from datetime import date, datetime, timedelta
from decimal import Decimal
from time import sleep
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from database.conexao import conectar
from indicadores.indicador import Indicador, IndicadorInfo

url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"
inicio = date(1999, 3, 5)
hoje = date.today()

retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
valor_anterior = None
total = 0

with conectar():
    cd_indicador = IndicadorInfo.get(IndicadorInfo.nm_indicador == "SELIC").cd_indicador

with requests.Session() as sessao:
    sessao.mount("https://", HTTPAdapter(max_retries=retry))

    while inicio <= hoje:
        fim = min(date(inicio.year, 12, 31), hoje)
        parametros = {
            "formato": "json",
            "dataInicial": inicio.strftime("%d/%m/%Y"),
            "dataFinal": fim.strftime("%d/%m/%Y")
        }

        resposta = sessao.get(url, params=parametros, timeout=60)
        resposta.raise_for_status()
        dados = resposta.json()

        registros = []

        for item in dados:
            valor = Decimal(item["valor"])

            if valor != valor_anterior:
                registros.append({
                    "cd_indicador": cd_indicador,
                    "valor": valor,
                    "status": True,
                    "dt_referencia": datetime.strptime(item["data"], "%d/%m/%Y").date()
                })

            valor_anterior = valor

        if registros:
            with conectar():
                Indicador.insert_many(registros).on_conflict_ignore().execute()

        total += len(registros)
        print(f"{inicio.year}: {len(registros)} alterações")

        inicio = fim + timedelta(days=1)
        sleep(1)

print(f"\nTotal de alterações consultadas: {total}")

1999: 12 alterações
2000: 5 alterações
2001: 6 alterações
2002: 6 alterações
2003: 9 alterações
2004: 6 alterações
2005: 9 alterações
2006: 8 alterações
2007: 6 alterações
2008: 4 alterações
2009: 5 alterações
2010: 3 alterações
2011: 8 alterações
2012: 7 alterações
2013: 6 alterações
2014: 5 alterações
2015: 5 alterações
2016: 2 alterações
2017: 8 alterações
2018: 2 alterações
2019: 4 alterações
2020: 5 alterações
2021: 7 alterações
2022: 5 alterações
2023: 4 alterações
2024: 6 alterações
2025: 4 alterações
2026: 5 alterações

Total de alterações consultadas: 162


In [12]:
carga_historica("SALARIO", 1619, 1994)


SALARIO 1994: 12 registros
SALARIO 1995: 12 registros
SALARIO 1996: 12 registros
SALARIO 1997: 12 registros
SALARIO 1998: 12 registros
SALARIO 1999: 12 registros
SALARIO 2000: 12 registros
SALARIO 2001: 12 registros
SALARIO 2002: 12 registros
SALARIO 2003: 12 registros
SALARIO 2004: 12 registros
SALARIO 2005: 12 registros
SALARIO 2006: 12 registros
SALARIO 2007: 12 registros
SALARIO 2008: 12 registros
SALARIO 2009: 12 registros
SALARIO 2010: 12 registros
SALARIO 2011: 12 registros
SALARIO 2012: 12 registros
SALARIO 2013: 12 registros
SALARIO 2014: 12 registros
SALARIO 2015: 12 registros
SALARIO 2016: 12 registros
SALARIO 2017: 12 registros
SALARIO 2018: 12 registros
SALARIO 2019: 12 registros
SALARIO 2020: 12 registros
SALARIO 2021: 12 registros
SALARIO 2022: 12 registros
SALARIO 2023: 12 registros
SALARIO 2024: 12 registros
SALARIO 2025: 12 registros
SALARIO 2026: 10 registros

SALARIO: 394 registros consultados
